# BoL Prediction Trial

This notebook tries the Book-of-Life setup after the temporal targets and pre-2000 features have been created.

Design:

- Input text: baseline Book-of-Life texts built from `XRND` and pre-2000 predictors
- Target: `later_persistent_delinquency_contact_2000_2020`
- Outcome window: 2000-2020

The notebook first runs a lightweight local text baseline. It also prepares an LLM prompt, but the LLM call is optional and only runs if an API key is available.

In [ ]:
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "BoL approach 2":
    APPROACH_DIR = cwd.parent
elif cwd.name == "BoL approach 2":
    APPROACH_DIR = cwd
else:
    APPROACH_DIR = cwd / "BoL approach 2"

TARGET_DIR = APPROACH_DIR / "data" / "targets"
FEATURE_DIR = APPROACH_DIR / "data" / "features"
BOOK_DIR = APPROACH_DIR / "data" / "books"
PRED_DIR = APPROACH_DIR / "data" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

BOOKS_JSON = BOOK_DIR / "baseline_pre2000_bol_books_sample_120.json"
TARGETS_CSV = TARGET_DIR / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
SAMPLE_IDS_CSV = FEATURE_DIR / "baseline_pre2000_bol_sample_ids_120.csv"
OUT_LOCAL_PREDS = PRED_DIR / "baseline_pre2000_bol_keyword_predictions_120.csv"

TARGET = "later_persistent_delinquency_contact_2000_2020"

## Load BoL Texts and Targets

In [ ]:
books = json.loads(BOOKS_JSON.read_text())
targets = pd.read_csv(TARGETS_CSV)
sample_ids = pd.read_csv(SAMPLE_IDS_CSV)

rows = []
for child_id, payload in books.items():
    rows.append({
        "C0000100": int(child_id),
        "text": payload["text"],
        "target_value_from_book": payload.get("target_value"),
    })

book_df = pd.DataFrame(rows)
model_df = book_df.merge(targets[["C0000100", TARGET]], on="C0000100", how="left")
model_df["text_length"] = model_df["text"].str.len()
model_df.head()

In [ ]:
print("Books:", len(model_df))
print("Missing target:", model_df[TARGET].isna().sum())
print(model_df[TARGET].value_counts(dropna=False))
print("Average text length:", int(model_df["text_length"].mean()))

## Local Keyword Baseline

This is not the final BoL LLM model. It is a quick smoke test using a transparent risk-word score. It runs without extra packages and checks whether the rendered pre-2000 books contain obvious signal related to the later target.

Because the sample has only 120 people, interpret results cautiously.

In [ ]:
risk_terms = {
    "disobedient": 1.0,
    "cheats": 1.0,
    "lies": 1.0,
    "argues": 1.0,
    "impulsive": 1.0,
    "temper": 1.0,
    "antisocial": 1.0,
    "external": 0.7,
    "fight": 1.5,
    "hurt someone": 1.5,
    "damaged": 1.4,
    "probation": 1.8,
    "corrections": 1.8,
    "jail": 1.8,
    "convicted": 1.6,
    "alcohol": 0.8,
    "marijuana": 0.8,
    "drug": 0.8,
}

protective_terms = {
    "no": 0.15,
    "school": 0.05,
    "achievement": 0.05,
}


def keyword_score(text):
    text = text.lower()
    score = 0.0
    for term, weight in risk_terms.items():
        score += weight * len(re.findall(re.escape(term), text))
    for term, weight in protective_terms.items():
        score -= weight * len(re.findall(r"\b" + re.escape(term) + r"\b", text))
    return score

model_df["keyword_score"] = model_df["text"].map(keyword_score)

# Convert scores to a probability-like scale using rank percentiles. This keeps
# the baseline transparent and avoids fitting a model on such a small sample.
model_df["probability"] = model_df["keyword_score"].rank(pct=True)
threshold = model_df["probability"].median()
model_df["prediction"] = (model_df["probability"] >= threshold).astype(int)
model_df["y_true"] = model_df[TARGET].astype(int)
model_df["correct"] = model_df["prediction"] == model_df["y_true"]

accuracy = model_df["correct"].mean()
confusion = pd.crosstab(model_df["y_true"], model_df["prediction"], rownames=["true"], colnames=["pred"], dropna=False)

print("Accuracy:", round(accuracy, 3))
print("Threshold:", round(float(threshold), 3))
print("Confusion matrix:")
display(confusion)

model_df[["C0000100", "y_true", "keyword_score", "probability", "prediction", "correct"]].head()

In [ ]:
pred_df = model_df[["C0000100", "y_true", "keyword_score", "probability", "prediction", "correct"]].copy()
pred_df.insert(1, "target", TARGET)
pred_df.insert(2, "model", "transparent_keyword_score_baseline")
pred_df.to_csv(OUT_LOCAL_PREDS, index=False)
print("Saved predictions to:", OUT_LOCAL_PREDS)
pred_df.head()

## Inspect Keyword Scores

This shows the strongest risk-score cases and lowest risk-score cases in the sample. It is only a diagnostic, not a causal interpretation.

In [ ]:
print("Highest keyword-score cases:")
display(model_df.sort_values("keyword_score", ascending=False)[["C0000100", "y_true", "keyword_score", "probability", "prediction"]].head(15))

print("Lowest keyword-score cases:")
display(model_df.sort_values("keyword_score", ascending=True)[["C0000100", "y_true", "keyword_score", "probability", "prediction"]].head(15))

## Prepare an LLM Prompt

The prompt below mirrors the BoL draft logic but uses the stricter baseline setup. The book text has already been restricted to pre-2000 predictors.

In [ ]:
def make_prompt(book_text):
    return f"""Read the Book of Life below and predict whether this person will show persistent delinquency/contact between 2000 and 2020.

The target equals 1 if the person has direct delinquency/contact indicators in at least two different survey years from 2000 to 2020. The indicators include physical fighting, hurting someone badly, damaging school property, probation, or being sentenced to corrections/jail/reform.

Important: the Book of Life only contains stable background variables and information observed before 2000.

Return only valid JSON with exactly these keys:
{{"prediction": 0 or 1, "probability": number between 0 and 1, "reason": "one short sentence"}}

Make `prediction` consistent with `probability`: prediction must be 1 if probability >= 0.50 and 0 otherwise.

Book of Life:
{book_text}
"""

example_row = model_df.iloc[0]
print(make_prompt(example_row["text"])[:2500])

## Optional OpenAI LLM Call

This cell is intentionally optional. It only runs if `OPENAI_API_KEY` is set. Otherwise it skips safely. For a full run, remove the small `head(5)` sample limit.

In [ ]:
RUN_LLM = False
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")

if RUN_LLM and os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    llm_rows = []
    for _, row in model_df.head(5).iterrows():
        response = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": "You are a careful research assistant making binary predictions from life-history text. Return compact JSON only."},
                {"role": "user", "content": make_prompt(row["text"])},
            ],
            max_output_tokens=180,
        )
        llm_rows.append({
            "C0000100": row["C0000100"],
            "target": TARGET,
            "model": MODEL,
            "y_true": int(row[TARGET]),
            "raw_response": response.output_text,
        })
    llm_df = pd.DataFrame(llm_rows)
    display(llm_df)
else:
    print("Skipping LLM call. Set RUN_LLM = True and provide OPENAI_API_KEY to run it.")

## Interpretation

Use this notebook as a first smoke test. If the local text baseline performs near chance, the rendered pre-2000 books may be too sparse/noisy or the target may need refinement. If it performs above chance, the next step is to run the same BoL texts through the intended LLM prediction workflow and compare against tabular baselines.